In [0]:
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice")
from utils.pipeline_audit import AuditManager
from datetime import datetime
import uuid

pipeline_name="Customer_Pipeline"
batch_id=str(uuid.uuid4())
print(batch_id)

def run_customer(notebook_path):
    notebook_name=notebook_path.split("/")[-1]
    print(notebook_name)
      
    try:
        start_time=datetime.now()

        print(f"Started : {notebook_name}") 
        print(f"Batch Id : {batch_id}")

        result = dbutils.notebook.run( 
                notebook_path, 
                timeout_seconds=3600, 
                arguments={ 
                    "batch_id": batch_id
             } 
        )

        end_time=datetime.now()
        duration=end_time-start_time
        print(f"Completed : {notebook_name}")
        print(f"Duration : {duration}")
        print(f"Status : {result}")
        AuditManager.write_audit(
            spark=spark,
            batch_id=batch_id,
            pipeline_name=pipeline_name,
            notebook_name=notebook_name,
            start_time=start_time,
            end_time=end_time,
            status="PASS",
            error_message=""
        )

    except Exception as e:
        print(f"failed note book execution : {notebook_name}")
        print(f"error: {str(e)}")
        AuditManager.write_audit(
            spark=spark,
            batch_id=batch_id,
            pipeline_name=pipeline_name,
            notebook_name=notebook_name,
            start_time=start_time,
            end_time=end_time,
            status="FAILED",
            error_message=str(e)
        )
        raise e



In [0]:
try:
    #run bronze notebook
    run_customer("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice/Note Book/p_bronze/bronze_load")

    # run silver notebook
    run_customer("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice/Note Book/p_silver/silver_curated")

    #run gold
    run_customer("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice/Note Book/p_gold/gold_transformed")

    print("Pipeline completed successfully")

except Exception as e:
    print(f"pipeline failed with error :{str(e)}")
    raise e